In [ ]:
import os
from alphagenome.data import gene_annotation
from alphagenome.data import genome
from alphagenome.data import transcript as transcript_utils
from alphagenome.visualization import plot_components
from alphagenome_research.model import dna_model
from alphagenome.models.variant_scorers import CenterMaskScorer
from alphagenome.models.variant_scorers import AggregationType
from alphagenome.models import variant_scorers
from alphagenome.models import dna_client
import numpy as np
import pandas as pd
import jax
import polars as pl
os.environ['CUDA_VISIBLE_DEVICES'] = '6'

In [1]:
jax.devices()

NameError: name 'jax' is not defined

In [ ]:
class AlphaGenomeScoreVariant:
    
    enformer_locus_to_ontology = {
        "F9": ["EFO:0001187"],
        "GP1BB": ["EFO:0002067"],
        "HBB": ["EFO:0002067"],
        "HBG1": ["EFO:0002067"],
        "HNF4A": ["UBERON:0002369"],
        "IRF4": ["CL:2000000", "CL:2000045", "EFO:0005720"],
        "IRF6": ["CL:0000312", "CL:1001606"],
        "LDLR": ["EFO:0001187"],
        "MSMB": ["UBERON:0002369"],
        "MYC": ["UBERON:0002369"],
        "PKLR": ["EFO:0002067"],
        "SORT1": ["EFO:0001187"],
        "TERT": ["UBERON:0002369"],
        "ZFAND3": ["CL:0002351", "UBERON:0001150", "UBERON:0001264"],
    }

    borzoi_locus_to_ontology = {
        "F9": ["EFO:0002067"],
        "GP1BB": ["EFO:0002067"],
        "HBB": ["EFO:0002067"],
        "HBG1": ["EFO:0002067"],
        "HNF4A": ["UBERON:0002113"],
        "IRF4": ["CL:2000000", "CL:2000045", "EFO:0005720"],
        "IRF6": ["CL:0000312", "CL:1001606"],
        "LDLR": ["UBERON:0002113"],
        "MSMB": ["UBERON:0002113"],
        "MYC": ["UBERON:0002113"],
        "PKLR": ["EFO:0002067"],
        "SORT1": ["EFO:0001187"],
        "TERT": ["UBERON:0002113"],
        "ZFAND3": ["CL:0002351", "UBERON:0001150", "UBERON:0001264"],
    }
    
    def __init__(self, model: dna_model.AlphaGenomeModel, ontology_to_use:str):
        self.model = model
        self.ontology_to_use = AlphaGenomeScoreVariant.borzoi_locus_to_ontology if ontology_to_use == 'borzoi' else AlphaGenomeScoreVariant.enformer_locus_to_ontology
        
        self.center_DNase_scorer = CenterMaskScorer(requested_output=dna_model.OutputType.DNASE, width=501, aggregation_type=AggregationType.DIFF_SUM)
        self.center_CAGE_scorer = CenterMaskScorer(requested_output=dna_model.OutputType.CAGE, width=501, aggregation_type=AggregationType.DIFF_SUM)
    
    def __call__(self, variantStrings:list, observed_change:list, elements:list):
        
        elements_clean = [el.split(' ')[0].split('.')[0].split('-')[0].split('rs')[0] for el in elements]
        ontologies = [self.ontology_to_use[el] for el in elements_clean]
        
        intervals = []
        variants = []
        for var in variantStrings:
            variant = genome.Variant.from_str(var)
            interval = variant.reference_interval.resize(2**20)
            intervals.append(interval)
            variants.append(variant)
        
        prediction = self.model.score_variants(
            intervals,
            variants,
            variant_scorers=[self.center_DNase_scorer],
            organism=dna_model.Organism.HOMO_SAPIENS,
            max_workers=20
            )
        df = variant_scorers.tidy_scores(prediction)[['variant_id', 'ontology_curie', 'raw_score']]
        df.variant_id = df.variant_id.astype(str)
        def select_by_ontology(struct, ontologies:set|list=["CL:0000312", "CL:1001606"]):
            ont_list = struct['ontology_curie']
            raw_scores = struct['raw_score']
            indices = [ont_list.index(ont) for ont in ontologies]
            raw_values = [raw_scores[idx] for idx in indices]
            return np.mean(raw_values) 
        scores = pl.DataFrame(df).group_by('variant_id') \
                        .agg(pl.col("*")) \
                        .with_columns(raw_score_averaged = pl.struct(pl.col.ontology_curie, pl.col.raw_score) \
                        .map_elements(select_by_ontology, return_dtype=pl.Float64, strategy='threading')) \
                        .sort(pl.col('variant_id').sort_by(pl.Series(variantStrings)))['raw_score_averaged'].to_list()
        
        
        return pl.DataFrame({'variants': variantStrings, 'Predicted': scores, 'Observed': observed_change})

In [ ]:
dataset = pl.read_csv('/home/jovyan/.cache/mpramnist/data/Kircher/Kircher_GRCh38_ALL.tsv', separator='\t', infer_schema_length=10000)
variant_expr = pl.lit('chr') + pl.col('Chromosome').cast(pl.String) + pl.lit(':') + (pl.col('Position') + 1).cast(pl.String) + pl.lit(':') + pl.col('Ref').str.replace('-', '') + pl.lit('>') + pl.col('Alt').str.replace('-', '')
dataset = dataset.with_columns(variant = variant_expr)
dataset = dataset.filter(pl.col.Tags.__gt__(10))
#dataset = dataset.with_columns(variant = variant_expr).filter(pl.col('Element').__eq__('PKLR-48h') | pl.col('Element').__eq__('PKLR-24h'))
#dataset = dataset.with_columns(variant = variant_expr).filter(pl.col('Element').__eq__('F9'))

In [4]:
dataset.filter(pl.col.Element.is_in({'BCL11A', 'IRF4', 'IRF6', 'MYCrs6983267', 'MYCrs11986220', 'RET', 'SORT1', 'SORT1-flip', 'SORT1.2', 'TCF7L2', 'UC88', 'ZFAND3', 'ZRSh-13', 'ZRSh-13h2'}))

Element,Cell_Type,Chromosome,Position,Ref,Alt,Tags,DNA,RNA,Value,P-Value,variant
str,str,str,i64,str,str,i64,i64,i64,f64,f64,str
"""BCL11A""","""HEL92.1.7""","""2""",60494939,"""C""","""-""",32,577,1345,-0.34,0.00546,"""chr2:60494940:C>"""
"""BCL11A""","""HEL92.1.7""","""2""",60494939,"""C""","""A""",146,2785,6772,-0.05,0.38889,"""chr2:60494940:C>A"""
"""BCL11A""","""HEL92.1.7""","""2""",60494939,"""C""","""G""",60,975,2436,-0.13,0.13721,"""chr2:60494940:C>G"""
"""BCL11A""","""HEL92.1.7""","""2""",60494939,"""C""","""T""",1084,8543,16057,-0.7,0.0,"""chr2:60494940:C>T"""
"""BCL11A""","""HEL92.1.7""","""2""",60494940,"""C""","""A""",596,9425,23430,-0.08,0.00413,"""chr2:60494941:C>A"""
…,…,…,…,…,…,…,…,…,…,…,…
"""ZRSh-13h2""","""NIH-3T3""","""7""",156791602,"""C""","""-""",1447,11717,34300,0.04,0.11224,"""chr7:156791603:C>"""
"""ZRSh-13h2""","""NIH-3T3""","""7""",156791602,"""C""","""A""",25,107,309,-0.17,0.33374,"""chr7:156791603:C>A"""
"""ZRSh-13h2""","""NIH-3T3""","""7""",156791602,"""C""","""G""",654,4748,14458,0.06,0.09092,"""chr7:156791603:C>G"""


In [5]:
dataset = dataset.filter(pl.col.Tags.__gt__(10))

In [7]:
model = dna_model.create_from_huggingface('all_folds', device=jax.devices()[0], organism_settings={dna_model.Organism.HOMO_SAPIENS: dna_model.OrganismSettings(fasta_path='/home/jovyan/.cache/mpramnist/data/Kircher/hg38.fa'), 
                                                                                                   dna_model.Organism.MUS_MUSCULUS: (
            dna_model.OrganismSettings()
        )})

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

In [8]:
varStrs = ['chr1:209815790:G>', 'chr1:209815790:G>A']
intervals = []
variants = []
for var in varStrs:
    variant = genome.Variant.from_str(var)
    interval = variant.reference_interval.resize(2**20)
    intervals.append(interval)
    variants.append(variant)
prediction = model.score_variants(
    intervals,
    variants,
    variant_scorers=[CenterMaskScorer(requested_output=dna_model.OutputType.DNASE, width=501, aggregation_type=AggregationType.DIFF_SUM)],
    organism=dna_model.Organism.HOMO_SAPIENS,
    max_workers=20
    )

  0%|          | 0/2 [00:00<?, ?it/s]

In [9]:
ontologies = {"CL:0000312", "CL:1001606"}
df = variant_scorers.tidy_scores(prediction)[['variant_id', 'ontology_curie', 'raw_score']]
df.variant_id = df.variant_id.astype(str)

In [15]:
for i in pl.DataFrame(df).iter_slices(n_rows=305):
    print(i)

shape: (305, 3)
┌───────────────────┬────────────────┬───────────┐
│ variant_id        ┆ ontology_curie ┆ raw_score │
│ ---               ┆ ---            ┆ ---       │
│ str               ┆ str            ┆ f32       │
╞═══════════════════╪════════════════╪═══════════╡
│ chr1:209815790:G> ┆ CL:0000047     ┆ 0.325851  │
│ chr1:209815790:G> ┆ CL:0000084     ┆ 0.322341  │
│ chr1:209815790:G> ┆ CL:0000115     ┆ 0.355545  │
│ chr1:209815790:G> ┆ CL:0000127     ┆ 0.35598   │
│ chr1:209815790:G> ┆ CL:0000134     ┆ 0.059502  │
│ …                 ┆ …              ┆ …         │
│ chr1:209815790:G> ┆ UBERON:0036149 ┆ 0.034735  │
│ chr1:209815790:G> ┆ UBERON:8300001 ┆ 0.117805  │
│ chr1:209815790:G> ┆ UBERON:8300002 ┆ 0.149036  │
│ chr1:209815790:G> ┆ UBERON:8300003 ┆ 0.258331  │
│ chr1:209815790:G> ┆ UBERON:8300004 ┆ 0.139153  │
└───────────────────┴────────────────┴───────────┘
shape: (305, 3)
┌────────────────────┬────────────────┬───────────┐
│ variant_id         ┆ ontology_curie ┆ raw_score

In [31]:
class AlphaGenomeCAGI5Benchmark:
    
    enformer_locus_to_ontology = {
        "F9": ["EFO:0001187"],
        "GP1BA": ["EFO:0002067"],
        "HBB": ["EFO:0002067"],
        "HBG1": ["EFO:0002067"],
        "HNF4A": ["UBERON:0002369"],
        "IRF4": ["CL:2000000", "CL:2000045", "EFO:0005720"],
        "IRF6": ["CL:0000312", "CL:1001606"],
        "LDLR": ["EFO:0001187"],
        "MSMB": ["UBERON:0002369"],
        "MYC": ["UBERON:0002369"],
        "PKLR": ["EFO:0002067"],
        "SORT1": ["EFO:0001187"],
        "TERT": ["UBERON:0002369"],
        "ZFAND3": ["CL:0002351", "UBERON:0001150", "UBERON:0001264"],
    }

    borzoi_locus_to_ontology = {
        "F9": ["EFO:0002067"],
        "GP1BA": ["EFO:0002067"],
        "HBB": ["EFO:0002067"],
        "HBG1": ["EFO:0002067"],
        "HNF4A": ["UBERON:0002113"],
        "IRF4": ["CL:2000000", "CL:2000045", "EFO:0005720"],
        "IRF6": ["CL:0000312", "CL:1001606"],
        "LDLR": ["UBERON:0002113"],
        "MSMB": ["UBERON:0002113"],
        "MYC": ["UBERON:0002113"],
        "PKLR": ["EFO:0002067"],
        "SORT1": ["EFO:0001187"],
        "TERT": ["UBERON:0002113"],
        "ZFAND3": ["CL:0002351", "UBERON:0001150", "UBERON:0001264"],
    }
    
    
    def __init__(
        self,
        model,
        locus_to_ontology,
        *,
        organism,
        scorer,
        interval_size=2**20,
        max_workers=20,
        score_rows_per_variant=305,
    ):
        self.model = model
        self.locus_to_ontology = locus_to_ontology
        self.organism = organism
        self.scorer = scorer
        self.interval_size = interval_size
        self.max_workers = max_workers
        self.score_rows_per_variant = score_rows_per_variant

    def make_intervals_and_variants(self, dataset: pl.DataFrame):
        intervals = []
        variants = []

        for var_str in dataset["variant"].to_list():
            variant = genome.Variant.from_str(var_str)
            interval = variant.reference_interval.resize(self.interval_size)
            variants.append(variant)
            intervals.append(interval)

        return intervals, variants

    def score_all_variants(self, dataset: pl.DataFrame) -> pl.DataFrame:
        intervals, variants = self.make_intervals_and_variants(dataset)

        prediction = self.model.score_variants(
            intervals,
            variants,
            variant_scorers=[self.scorer],
            organism=self.organism,
            max_workers=self.max_workers,
        )

        scores = variant_scorers.tidy_scores(prediction)[
            ["variant_id", "ontology_curie", "raw_score"]
        ]
        scores.variant_id = scores.variant_id.astype(str)

        if not isinstance(scores, pl.DataFrame):
            scores = pl.from_pandas(scores)

        scores = scores.with_columns(
            pl.col("variant_id").cast(pl.Utf8),
            pl.col("ontology_curie").cast(pl.Utf8),
            pl.col("raw_score").cast(pl.Float64),
        )

        expected_rows = len(dataset) * self.score_rows_per_variant
        if scores.height != expected_rows:
            print(
                f"Warning: expected {expected_rows} score rows "
                f"({len(dataset)} variants * {self.score_rows_per_variant}), "
                f"got {scores.height}."
            )

        return scores

    def parse_scores(
        self,
        dataset: pl.DataFrame,
        scores: pl.DataFrame,
        *,
        output_col="alphagenome_score",
    ) -> pl.DataFrame:
        ontology_map = pl.DataFrame(
            [
                {"Element": element, "ontology_curie": ontology}
                for element, ontologies in self.locus_to_ontology.items()
                for ontology in ontologies
            ]
        )
        dataset = dataset.with_columns(Element = pl.col.Element.map_elements(lambda el: el.split('.')[0].split('-')[0].split('rs')[0]))
        dataset_with_row_id = dataset.with_row_index("__row_id__")

        wanted_scores = (
            dataset_with_row_id
            .select("__row_id__", "Element", "variant")
            .join(ontology_map, on="Element", how="left")
        )
        
        wanted_scores = wanted_scores.filter(~pl.col("ontology_curie").is_null())

        #missing = wanted_scores.filter(pl.col("ontology_curie").is_null())
        #if missing.height:
        #    missing_elements = missing["Element"].unique().to_list()
        #    raise ValueError(f"No ontology mapping for elements: {missing_elements}")

        parsed = (
            wanted_scores
            .join(
                scores,
                left_on=["variant", "ontology_curie"],
                right_on=["variant_id", "ontology_curie"],
                how="left",
            )
            .group_by("__row_id__")
            .agg(
                pl.col("raw_score").mean().alias(output_col),
                pl.col("ontology_curie").alias("used_ontologies"),
                pl.col("raw_score").alias("ontology_scores"),
            )
        )

        missing_scores = parsed.filter(pl.col(output_col).is_null())
        if missing_scores.height:
            bad_rows = missing_scores["__row_id__"].to_list()[:10]
            raise ValueError(f"Missing AlphaGenome scores for dataset rows: {bad_rows}")

        return (
            dataset_with_row_id
            .join(parsed, on="__row_id__", how="left")
            .drop("__row_id__")
        )

    def run(self, dataset: pl.DataFrame):
        scores = self.score_all_variants(dataset)
        result = self.parse_scores(dataset, scores)
        return result, scores

In [32]:
sample = dataset.sample(10)

In [33]:
bench = AlphaGenomeCAGI5Benchmark(
    model=model,
    locus_to_ontology=AlphaGenomeCAGI5Benchmark.enformer_locus_to_ontology,
    organism=dna_model.Organism.HOMO_SAPIENS,
    scorer=CenterMaskScorer(
        requested_output=dna_model.OutputType.DNASE,
        width=501,
        aggregation_type=AggregationType.DIFF_SUM,
    ),
    max_workers=20,
)

result, raw_scores = bench.run(sample)

  0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
result

Element,Cell_Type,Chromosome,Position,Ref,Alt,Tags,DNA,RNA,Value,P-Value,variant,alphagenome_score,used_ontologies,ontology_scores
str,str,str,i64,str,str,i64,i64,i64,f64,f64,str,f64,list[str],list[f64]
"""BCL11A""","""HEL92.1.7""","""2""",60495060,"""T""","""A""",669,13762,34420,0.01,0.79977,"""chr2:60495061:T>A""",null,null,null
"""IRF6""","""HaCaT""","""1""",209815803,"""G""","""C""",46,13258,13695,-0.02,0.75244,"""chr1:209815804:G>C""",-77.077209,"[""CL:0000312"", ""CL:1001606""]","[-51.559692, -102.594727]"
"""IRF6""","""HaCaT""","""1""",209816049,"""G""","""A""",176,53887,62579,0.04,0.23855,"""chr1:209816050:G>A""",148.298523,"[""CL:0000312"", ""CL:1001606""]","[143.187866, 153.40918]"
"""MSMB""","""HEK293T""","""10""",46046606,"""C""","""T""",4984,52557,108736,0.0,0.90686,"""chr10:46046607:C>T""",-0.834774,"[""UBERON:0002369""]",[-0.834774]
"""SORT1""","""HepG2""","""1""",109275060,"""T""","""G""",26,12470,18923,0.29,0.00063,"""chr1:109275061:T>G""",102.906128,"[""EFO:0001187""]",[102.906128]
"""SORT1""","""HepG2""","""1""",109274834,"""A""","""T""",347,35447,14225,-0.06,0.1873,"""chr1:109274835:A>T""",-103.665283,"[""EFO:0001187""]",[-103.665283]
"""SORT1""","""HepG2""","""1""",109275128,"""C""","""T""",318,35899,18745,0.4,0.0,"""chr1:109275129:C>T""",27.655029,"[""EFO:0001187""]",[27.655029]
"""TERT""","""HEK293T""","""5""",1295142,"""G""","""A""",793,56532,64888,0.08,0.00859,"""chr5:1295143:G>A""",8.263641,"[""UBERON:0002369""]",[8.263641]
"""UC88""","""Neuro-2a""","""2""",161238836,"""C""","""A""",295,51255,63960,-0.21,0.0,"""chr2:161238837:C>A""",null,null,null


: 

In [96]:
["CL:0000047", "CL:0000084", "UBERON:8300004"]

1

In [ ]:
pl.DataFrame(df).group_by('variant_id').agg(pl.col("*")).map_rows()

In [70]:
pl.DataFrame(df.iterrows())

column_0,column_1
i64,object
0,"variant_id chr1:209815790:G> ontology_curie CL:0000047 raw_score 0.332077 Name: 0, dtype: object"
1,"variant_id chr1:209815790:G> ontology_curie CL:0000084 raw_score 0.325681 Name: 1, dtype: object"
2,"variant_id chr1:209815790:G> ontology_curie CL:0000115 raw_score 0.346283 Name: 2, dtype: object"
3,"variant_id chr1:209815790:G> ontology_curie CL:0000127 raw_score 0.370087 Name: 3, dtype: object"
4,"variant_id chr1:209815790:G> ontology_curie CL:0000134 raw_score 0.060936 Name: 4, dtype: object"
…,…
605,"variant_id chr1:209815790:G>A ontology_curie UBERON:0036149 raw_score -0.573956 Name: 605, dtype: object"
606,"variant_id chr1:209815790:G>A ontology_curie UBERON:8300001 raw_score -0.80757 Name: 606, dtype: object"
607,"variant_id chr1:209815790:G>A ontology_curie UBERON:8300002 raw_score -0.651495 Name: 607, dtype: object"


In [16]:
dataset

Element,Cell_Type,Chromosome,Position,Ref,Alt,Tags,DNA,RNA,Value,P-Value,variant
str,str,str,i64,str,str,i64,i64,i64,f64,f64,str
"""BCL11A""","""HEL92.1.7""","""2""",60494939,"""C""","""-""",32,577,1345,-0.34,0.00546,"""chr2:60494940:C>"""
"""BCL11A""","""HEL92.1.7""","""2""",60494939,"""C""","""A""",146,2785,6772,-0.05,0.38889,"""chr2:60494940:C>A"""
"""BCL11A""","""HEL92.1.7""","""2""",60494939,"""C""","""G""",60,975,2436,-0.13,0.13721,"""chr2:60494940:C>G"""
"""BCL11A""","""HEL92.1.7""","""2""",60494939,"""C""","""T""",1084,8543,16057,-0.7,0.0,"""chr2:60494940:C>T"""
"""BCL11A""","""HEL92.1.7""","""2""",60494940,"""C""","""A""",596,9425,23430,-0.08,0.00413,"""chr2:60494941:C>A"""
…,…,…,…,…,…,…,…,…,…,…,…
"""ZRSh-13h2""","""NIH-3T3""","""7""",156791601,"""G""","""T""",1676,12030,37310,0.1,0.00001,"""chr7:156791602:G>T"""
"""ZRSh-13h2""","""NIH-3T3""","""7""",156791602,"""C""","""-""",1447,11717,34300,0.04,0.11224,"""chr7:156791603:C>"""
"""ZRSh-13h2""","""NIH-3T3""","""7""",156791602,"""C""","""A""",25,107,309,-0.17,0.33374,"""chr7:156791603:C>A"""


In [4]:
predictor = AlphaGenomeScoreVariant(ontology_to_use='enformer', model = model)

In [12]:
output = predictor(variantStrings = dataset['variant'].to_list(), observed_change=dataset['Value'].to_list(), ontology="EFO:0002067")

  0%|          | 0/950 [00:00<?, ?it/s]

In [9]:
#DNase result on PKLR, filtered, tags >10
output[['Predicted', 'Observed']].corr()

Predicted,Observed
f64,f64
1.0,0.79778
0.79778,1.0


In [18]:
#DNase result on PKLR, filtered, tags >10, pvalue < 0.1
output[['Predicted', 'Observed']].corr()

Predicted,Observed
f64,f64
1.0,0.84749
0.84749,1.0


In [ ]:
#DNase result on PKLR, no filtering
output[['Predicted', 'Observed']].corr()

Predicted,Observed
f64,f64
1.0,0.585635
0.585635,1.0


In [13]:
#DNase result on F9, tags > 10
output[['Predicted', 'Observed']].corr()

Predicted,Observed
f64,f64
1.0,0.387356
0.387356,1.0


In [ ]:
#DNase result on F9
output[['Predicted', 'Observed']].corr()

Predicted,Observed
f64,f64
1.0,0.319857
0.319857,1.0


In [ ]:
import numpy as np
import polars as pl

def compute_element_correlations(
    result: pl.DataFrame,
    *,
    element_col="Element",
    target_col="Value",
    pred_col="alphagenome_score",
    method="pearson",
    min_n=2,
):
    """
    Returns:
        dict[element] = correlation between experimental Value and AlphaGenome score
    """

    if not isinstance(result, pl.DataFrame):
        result = pl.from_pandas(result)

    df = (
        result
        .select(element_col, target_col, pred_col)
        .drop_nulls([element_col, target_col, pred_col])
    )

    correlations = {}

    for key, group in df.partition_by(element_col, as_dict=True).items():
        element = key[0] if isinstance(key, tuple) else key

        x = group[target_col].cast(pl.Float64).to_numpy()
        y = group[pred_col].cast(pl.Float64).to_numpy()

        if len(x) < min_n or np.std(x) == 0 or np.std(y) == 0:
            correlations[element] = np.nan
            continue

        if method == "pearson":
            corr = np.corrcoef(x, y)[0, 1]

        elif method == "spearman":
            x_rank = pl.Series(x).rank("average").to_numpy()
            y_rank = pl.Series(y).rank("average").to_numpy()
            corr = np.corrcoef(x_rank, y_rank)[0, 1]

        else:
            raise ValueError("method must be 'pearson' or 'spearman'")

        correlations[element] = float(corr)

    return correlations